# 2026-06-17 Day 5: Generation, Sampling, And MoE Extension

Goal: understand autoregressive generation, sampling controls, top-p motivation, and how MoE routing differs from token sampling.


## How To Use This Reading Notebook

Use this before touching implementation for the day. The goal is not to memorize the paper. The goal is to extract the model of the system you need for today's code and notes.

Workflow:

1. Read only the assigned sections.
2. Fill the mini-lecture answer in your own words.
3. Open the repo files listed in the roadmap.
4. Run the listed checks or record why they skip.
5. Write a short exit ticket before moving on.


## Assigned Reading

| Source | Exact Target | Why It Matters Today |
| --- | --- | --- |
| [Hugging Face generation docs](https://huggingface.co/docs/transformers/main_classes/text_generation) | `generate`, `GenerationConfig`, `temperature`, `top_k`, `top_p`, `use_cache` | Establishes the vocabulary sampling controls used in generation APIs. |
| [The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751) | Abstract and introduction | Motivates nucleus sampling and the problem with unreliable probability tails. |


## Reading Summary

### Hugging Face Generation Docs

Generation is controlled by a configuration: output length, sampling mode, cache use, and logit manipulation settings. `max_new_tokens` controls the number of generated tokens independent of prompt length. `temperature` changes distribution sharpness. `top_k` keeps a fixed number of candidate vocabulary tokens. `top_p` keeps the smallest set of high-probability tokens whose cumulative probability reaches the threshold.

Key takeaway for implementation: generation is a loop around the model's final-position logits. Sampling controls act on vocabulary logits, not hidden states.

### Neural Text Degeneration

The paper argues that pure likelihood maximization can produce bland or repetitive text, while unrestricted sampling can draw from the unreliable tail of the distribution. Nucleus sampling trims the tail dynamically by keeping a probability-mass prefix rather than a fixed number of tokens.

Key takeaway for implementation: top-p is adaptive. It may keep many tokens when the model is uncertain and only a few when the model is confident.


## Diagram Anchor

![Sampling filters](../../assets/figures/sampling_filters.svg)

What to notice: top-k and top-p filter vocabulary candidates before sampling.

![Generation loop](../../assets/figures/generation_loop.svg)

What to notice: only the final-position logits choose the next token.

![MoE routing](../../assets/figures/moe_routing.svg)

What to notice: MoE top-k selects experts for hidden states, not vocabulary tokens for generation.


## Repo Connection

Open these files after reading:

| File | What To Find |
| --- | --- |
| `scripts/generate.py` | CLI args, tokenizer load, checkpoint load, `model.generate`. |
| `src/moe_transformer_lab/trainable/model.py` | `generate`, top-k filter, temperature scaling, `MoEFeedForward`. |
| `tests/test_swappable_ffn.py` | Dense/MoE feed-forward contract tests. |


## Sampling Summary

| Control | What It Does | Failure Mode |
| --- | --- | --- |
| greedy | picks argmax | deterministic but repetitive or bland |
| temperature | rescales logits | too high gets noisy, too low collapses |
| top-k | keeps fixed `k` tokens | fixed cutoff may be too narrow or too broad |
| top-p | keeps probability-mass prefix | needs careful sorting, masking, and renormalization |


## Mini-Lecture Answer Seed

Generation crops the visible context, runs a forward pass, reads `logits[:, -1, :]`, applies sampling controls, samples or chooses one token, appends it, and repeats. Vocabulary top-k/top-p operate on next-token probabilities. MoE router top-k operates inside the model on token representations and chooses expert MLPs. They are unrelated uses of the phrase top-k.


## Active Recall

1. Why does generation use only `logits[:, -1, :]`?
2. Why is top-p adaptive while top-k is fixed-size?
3. What does `max_new_tokens` control?
4. Why does MoE not require a separate generation loop?

## Exit Ticket

Write planned top-p pseudocode and one deterministic generation test idea.
